##Reading Stream

In [0]:
(spark.readStream
    .table("books")
    .createOrReplaceTempView("books_streaming_tmp_vw")
)

The temporary view created above is a **"Streaming" Temporary view** that allows to apply most transformations in SQL the same way as we would with the static data

##Displaying Stream Data

In [0]:
import time
books_streaming_df = spark.sql("SELECT * FROM books_streaming_tmp_vw")
display(books_streaming_df, checkpointLocation = f"/Volumes/demoworkspace/default/bookstore_checkpoints/tmp/books_streaming_{time.time()}")

##Applying Transformation

In [0]:
author_count_df=spark.sql("select author,count(book_id) as total_books from books_streaming_tmp_vw group by author")
display(author_count_df, checkpointLocation = f"/Volumes/demoworkspace/default/bookstore_checkpoints/tmp/author_count_{time.time()}")

##Unsupported Operations

**Sorting** is not supported on streaming DataFrames/Datasets.You can use advanced methods like **Windowing and Watermarking** to achieve Such operations

In [0]:
sorted_books_df = books_streaming_df.orderBy("author")

(sorted_books_df.writeStream
                .option("checkpointLocation", f"/Volumes/demoworkspace/default/bookstore_checkpoints/sorted_books")
                .trigger(availableNow=True)
                .format("console")
                .start()
)

##Persisting Streaming data

In [0]:
%sql
create or replace temporary view author_counts_tmp_vw as(
    select author,count(book_id) as total_books
    from books_streaming_tmp_vw
    group by author
)

In [0]:
(spark.table("author_counts_tmp_vw")
      .writeStream
      .trigger(processingTime='4 seconds')
      .outputMode("Complete")
      .option("checkpointLocation", f"/Volumes/demoworkspace/default/bookstore_checkpoints/author_counts")
      .toTable("author_counts")
)

Using **spark.table()** to load data from a streaming temporary view back to a DF

In [0]:
%sql
select * from author_counts

##Adding new Data

In [0]:
%sql
INSERT INTO books
values ("B19", "Introduction to Modeling and Simulation", "Mark W. Spong", "Computer Science", 25),
        ("B20", "Robot Modeling and Control", "Mark W. Spong", "Computer Science", 30),
        ("B21", "Turing's Vision: The Birth of Computer Science", "Chris Bernhardt", "Computer Science", 35)

In [0]:
%sql
select * from author_counts

##Streaming in Batch Mode

In [0]:
%sql
INSERT INTO books
values ("B16", "Hands-On Deep Learning Algorithms with Python", "Sudharsan Ravichandiran", "Computer Science", 25),
        ("B17", "Neural Network Methods in Natural Language Processing", "Yoav Goldberg", "Computer Science", 30),
        ("B18", "Understanding digital signal processing", "Richard Lyons", "Computer Science", 35)

In [0]:
(spark.table("author_counts_tmp_vw")
    .writeStream
    .trigger(availableNow=True)
    .outputMode("complete")
    .option("checkpointLocation", f"/Volumes/demoworkspace/default/bookstore_checkpoints/author_counts")
    .toTable("author_counts")
    .awaitTermination()
)

**awaitTermination()** menthod blocks the executuion of any cell in this notebook until the incremental batch's write has succeeded 

With the **availableNow()** trigger option, the query runs in a batch mode.It is executed to process all the available data and then stop on its own

In [0]:
%sql
select 
    *
from
    author_counts